In [1]:
import json
from pathlib import Path
import pandas as pd
from shapely.geometry import Polygon
from shapely.validation import make_valid

In [2]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [3]:
def to_polygon(points):
    poly = Polygon(points)
    if not poly.is_valid:
        poly = make_valid(poly)
    if poly.is_empty or poly.area == 0:
        return None
    return poly


def extract_polygons_from_json(data):
    polys = []
    for pts in data.get("polys", []):
        poly = to_polygon(pts)
        if poly is not None:
            polys.append(poly)
    return polys


def intersection_area(p1, p2):
    inter = p1.intersection(p2)
    return inter.area if not inter.is_empty else 0.0


def union_area(p1, p2):
    uni = p1.union(p2)
    return uni.area if not uni.is_empty else 0.0


def iou(p1, p2):
    inter = intersection_area(p1, p2)
    uni = union_area(p1, p2)
    return inter / uni if uni > 0 else 0.0


def gt_coverage(jap_poly, gt_poly):
    inter = intersection_area(jap_poly, gt_poly)
    return inter / gt_poly.area if gt_poly.area > 0 else 0.0


def pred_coverage(jap_poly, gt_poly):
    inter = intersection_area(jap_poly, gt_poly)
    return inter / jap_poly.area if jap_poly.area > 0 else 0.0


def dice(p1, p2):
    inter = intersection_area(p1, p2)
    denom = p1.area + p2.area
    return (2.0 * inter) / denom if denom > 0 else 0.0


def best_jap_match_per_gt(jap_polys, gt_polys):
    """
    For each GT polygon, keep only the single Japanese polygon
    with the highest IoU.
    """
    matches = []

    for gt_idx, gt_poly in enumerate(gt_polys):
        best_jap_idx = None
        best_iou = -1.0
        best_jap_poly = None

        for jap_idx, jap_poly in enumerate(jap_polys):
            score = iou(jap_poly, gt_poly)
            if score > best_iou:
                best_iou = score
                best_jap_idx = jap_idx
                best_jap_poly = jap_poly

        if best_jap_poly is None:
            continue

        matches.append({
            "gt_idx": gt_idx,
            "jap_idx": best_jap_idx,
            "iou": best_iou,
            "gt_coverage": gt_coverage(best_jap_poly, gt_poly),
            "pred_coverage": pred_coverage(best_jap_poly, gt_poly),
            "dice": dice(best_jap_poly, gt_poly),
        })

    return matches


def summarize(matches):
    if not matches:
        return {
            "num_gt_boxes": 0,
            "num_matched_gt": 0,
            "mean_iou": 0.0,
            "median_iou": 0.0,
            "iou@0.3": 0.0,
            "iou@0.5": 0.0,
            "iou@0.7": 0.0,
            "mean_gt_coverage": 0.0,
            "mean_pred_coverage": 0.0,
            "mean_dice": 0.0,
        }

    df = pd.DataFrame(matches)

    return {
        "num_matched_gt": len(df),
        "mean_iou": df["iou"].mean(),
        "median_iou": df["iou"].median(),
        "iou@0.3": (df["iou"] >= 0.3).mean(),
        "iou@0.5": (df["iou"] >= 0.5).mean(),
        "iou@0.7": (df["iou"] >= 0.7).mean(),
        "mean_gt_coverage": df["gt_coverage"].mean(),
        "mean_pred_coverage": df["pred_coverage"].mean(),
        "mean_dice": df["dice"].mean(),
    }


def evaluate_file(jap_json_path, gt_json_path):
    jap_data = load_json(jap_json_path)
    gt_data = load_json(gt_json_path)

    jap_polys = extract_polygons_from_json(jap_data)
    gt_polys = extract_polygons_from_json(gt_data)

    matches = best_jap_match_per_gt(jap_polys, gt_polys)
    summary = summarize(matches)

    return {
        "file_name": Path(jap_json_path).name,
        "num_jap_polys": len(jap_polys),
        "num_gt_polys": len(gt_polys),
        "num_matched_gt": summary["num_matched_gt"],
        "mean_iou": summary["mean_iou"],
        "median_iou": summary["median_iou"],
        "iou@0.3": summary["iou@0.3"],
        "iou@0.5": summary["iou@0.5"],
        "iou@0.7": summary["iou@0.7"],
        "mean_gt_coverage": summary["mean_gt_coverage"],
        "mean_pred_coverage": summary["mean_pred_coverage"],
        "mean_dice": summary["mean_dice"],
    }


def evaluate_dirs_to_csv(jap_dir, gt_dir, out_csv):
    jap_dir = Path(jap_dir)
    gt_dir = Path(gt_dir)

    rows = []

    for jap_file in sorted(jap_dir.glob("*.json")):
        gt_file = gt_dir / jap_file.name

        if not gt_file.exists():
            rows.append({
                "file_name": jap_file.name,
                "num_jap_polys": None,
                "num_gt_polys": None,
                "num_matched_gt": None,
                "mean_iou": None,
                "median_iou": None,
                "iou@0.3": None,
                "iou@0.5": None,
                "iou@0.7": None,
                "mean_gt_coverage": None,
                "mean_pred_coverage": None,
                "mean_dice": None,
                "error": "Missing GT file",
            })
            continue

        try:
            row = evaluate_file(jap_file, gt_file)
            row["error"] = None
            rows.append(row)
        except Exception as e:
            rows.append({
                "file_name": jap_file.name,
                "num_jap_polys": None,
                "num_gt_polys": None,
                "num_matched_gt": None,
                "mean_iou": None,
                "median_iou": None,
                "iou@0.3": None,
                "iou@0.5": None,
                "iou@0.7": None,
                "mean_gt_coverage": None,
                "mean_pred_coverage": None,
                "mean_dice": None,
                "error": str(e),
            })

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df

In [4]:
jap_dir = r"outputs\jap_ocr_otsu\json"
gt_dir = r"outputs\en_ocr_otsu\json"
out_csv = r"ocr_overlap_metrics.csv"

df = evaluate_dirs_to_csv(jap_dir, gt_dir, out_csv)
print(df)

   file_name  num_jap_polys  num_gt_polys  num_matched_gt  mean_iou  \
0  pg_1.json              9             8               8  0.070036   
1  pg_2.json             16            12              12  0.106084   
2  pg_3.json             18            13              13  0.301888   
3  pg_4.json             12             8               8  0.092967   
4  pg_5.json             13            12              12  0.104341   
5  pg_6.json             18            16              16  0.047808   

   median_iou   iou@0.3   iou@0.5   iou@0.7  mean_gt_coverage  \
0    0.020895  0.000000  0.000000  0.000000          0.150694   
1    0.009681  0.083333  0.083333  0.000000          0.143582   
2    0.318135  0.538462  0.153846  0.076923          0.466745   
3    0.025918  0.125000  0.125000  0.000000          0.143704   
4    0.051953  0.083333  0.083333  0.000000          0.269051   
5    0.006771  0.000000  0.000000  0.000000          0.129019   

   mean_pred_coverage  mean_dice error  
0    